In [16]:
import dspy

# 1. Apagamos el caché para obligar al sistema a calcular los tokens reales
dspy.configure_cache(enable_disk_cache=False, enable_memory_cache=False)

# 2. Configuramos el modelo con la URL explícita de tu contenedor Ollama
# NOTA: Si tu script corre en tu máquina (fuera de Docker), cambia 'http://ollama:11434' por 'http://localhost:11434'
lm = dspy.LM(
    model='ollama/llama3.2:latest', 
    api_base='http://ollama:11434'
)

# 3. Activamos el rastreo de tokens de forma global
dspy.configure(lm=lm, track_usage=True)


In [17]:
# 4. Definimos la firma simple
class GenerarChiste(dspy.Signature):
    """Genera un chiste corto sobre programación."""
    tema: str = dspy.InputField()
    chiste: str = dspy.OutputField()

# 5. Creamos el predictor y ejecutamos
predictor = dspy.Predict(GenerarChiste)
resultado = predictor(tema="Python")

print("--- RESPUESTA ---")
print(resultado.chiste)

print("\n--- TOKENS CONSUMIDOS ---")
print(resultado.get_lm_usage())

--- RESPUESTA ---
¿Por qué la función no quería ir al trabajo? Porque siempre tenía que dividir su tiempo.

--- TOKENS CONSUMIDOS ---
{'ollama/llama3.2:latest': {'completion_tokens': 78, 'prompt_tokens': 325, 'total_tokens': 403, 'completion_tokens_details': None, 'prompt_tokens_details': None}}


In [18]:
import dspy
from typing import Literal

# 1. Configuración de Ollama (la que ya te funciona)
dspy.configure_cache(enable_disk_cache=False, enable_memory_cache=False)
lm = dspy.LM(
    model='ollama/llama3.2:latest', 
    api_base='http://ollama:11434'
)
dspy.configure(lm=lm, track_usage=True)

# --- Tu lógica del Agente de Alquiler ---
TerminoGlosario = Literal["deposito", "expensas", "rescision", "preaviso"]

def consultar_glosario(termino: TerminoGlosario) -> str:
    """Busca definiciones de términos de contratos de alquiler."""
    glosario: dict[TerminoGlosario, str] = {
        "deposito": "Suma que garantiza el contrato; suele devolverse al finalizar si no hay deudas ni daños.",
        "expensas": "Gastos del edificio. Ordinarias = mantenimiento habitual; extraordinarias = obras mayores.",
        "rescision": "Finalización anticipada del contrato por una de las partes.",
        "preaviso": "Aviso previo obligatorio antes de terminar o modificar el contrato.",
    }
    return glosario.get(termino, "Término no encontrado en el glosario.")

class AsistenteAlquiler(dspy.Signature):
    """Responde dudas sobre cláusulas de alquiler usando el glosario si hace falta."""
    pregunta: str = dspy.InputField()
    respuesta: str = dspy.OutputField()

agente = dspy.ReAct(AsistenteAlquiler, tools=[consultar_glosario])

# Ejecutamos el agente
r = agente(pregunta="Según la cláusula, ¿qué significa que el depósito sea de tres meses?")

print("--- RESPUESTA FINAL ---")
print(r.respuesta)


# --- PASO CLAVE: DESGLOSAR LOS TOKENS DE CADA PROMPT ---
print("\n--- DESGLOSE DE TOKENS POR CADA LLAMADA (PROMPT) ---")

# dspy.settings.lm.history guarda la lista de todas las interacciones de esta sesión
for i, llamada in enumerate(dspy.settings.lm.history):
    print(f"\n📢 [Llamada {i + 1}]")
    
    # Extraemos la metadata de la respuesta de LiteLLM / Ollama
    metadata = llamada.get('response', None)
    
    if metadata and hasattr(metadata, 'usage'):
        usage = metadata.usage
        print(f"  🔹 Tokens del Prompt (Entrada): {usage.prompt_tokens}")
        print(f"  🔹 Tokens de la Respuesta (Salida): {usage.completion_tokens}")
        print(f"  🔹 Total de esta llamada: {usage.total_tokens}")
    else:
        print("  ⚠️ No se pudieron extraer los tokens de esta llamada específica.")
        
    # Opcional: Si querés ver una vista previa del texto de ese prompt para identificarlo:
    # print(f"  📝 Inicio del prompt enviado: {llamada['prompt'][:100]}...")


--- RESPUESTA FINAL ---
El depósito de tres meses implica que el arrendatario debe pagar una cantidad equivalente al monto del depósito dentro de ese plazo.

--- DESGLOSE DE TOKENS POR CADA LLAMADA (PROMPT) ---

📢 [Llamada 1]
  🔹 Tokens del Prompt (Entrada): 671
  🔹 Tokens de la Respuesta (Salida): 72
  🔹 Total de esta llamada: 743

📢 [Llamada 2]
  🔹 Tokens del Prompt (Entrada): 660
  🔹 Tokens de la Respuesta (Salida): 93
  🔹 Total de esta llamada: 753

📢 [Llamada 3]
  🔹 Tokens del Prompt (Entrada): 1022
  🔹 Tokens de la Respuesta (Salida): 98
  🔹 Total de esta llamada: 1120

📢 [Llamada 4]
  🔹 Tokens del Prompt (Entrada): 1011
  🔹 Tokens de la Respuesta (Salida): 80
  🔹 Total de esta llamada: 1091

📢 [Llamada 5]
  🔹 Tokens del Prompt (Entrada): 1358
  🔹 Tokens de la Respuesta (Salida): 72
  🔹 Total de esta llamada: 1430

📢 [Llamada 6]
  🔹 Tokens del Prompt (Entrada): 1347
  🔹 Tokens de la Respuesta (Salida): 69
  🔹 Total de esta llamada: 1416

📢 [Llamada 7]
  🔹 Tokens del Prompt (Entra

Esta salida significa que tu agente dspy.ReAct entró en un bucle de razonamiento (loop) de 8 pasos antes de poder darte la respuesta final, lo que disparó exponencialmente el consumo de tokens.
En lugar de hacer una sola pregunta y dar una respuesta, el modelo Llama 3.2 estuvo intentando resolver el problema de forma interactiva en su mente ("Pensamiento -> Acción -> Observación") durante 8 iteraciones consecutivas.
Aquí te desgloso los 3 puntos clave de lo que está pasando:

1. El efecto "Bola de Nieve" de los Tokens

Fijate cómo los tokens de entrada aumentan en escalera: pasás de 671 en la Llamada 1 a 1,358 en la Llamada 5. Esto pasa porque en cada llamada de un agente ReAct, DSPy vuelve a enviar todo el historial acumulado de los pasos anteriores (lo que pensó antes, la herramienta que usó y lo que la herramienta le respondió). El prompt se vuelve cada vez más y más pesado.

2. Llama 3.2 se "perdió" en el razonamiento

Los modelos más chicos (como Llama 3.2 de 3B parámetros) suelen tener dificultades para seguir el formato estricto que exige un agente ReAct. Lo que pasó acá es que el modelo:
• O bien no entendió cómo formatear correctamente el llamado a la función consultar_glosario.
• O bien la herramienta le devolvía el texto y el modelo no sabía cómo cerrarlo, por lo que seguía intentando "razonar" una y otra vez en las llamadas 3, 4, 5 y 6 hasta que chocó con el límite de intentos del agente.

3. El costo total oculto

Si sumamos las 8 llamadas interconectadas, para responder una sola pregunta el script consumió:
• Tokens totales: ~8,700 tokens.
Para un modelo local en tu máquina esto solo significa unos segundos de demora, pero si estuvieras usando una API paga como OpenAI, una sola pregunta te habría costado 15 veces más de lo necesario.

¿Cómo solucionarlo para que consuma menos?

Si querés que resuelva la consulta en una sola llamada (un solo prompt de ~600 tokens) sin dar tantas vueltas, tenés dos opciones:
Opción A: Bajar el límite de pasos del agente
Por defecto, dspy.ReAct intenta hasta 5 o más iteraciones si está confundido. Podés obligarlo a detenerse antes con el parámetro max_iters:

In [19]:
agente = dspy.ReAct(AsistenteAlquiler, tools=[consultar_glosario], max_iters=2)

In [20]:
class AsistenteConContexto(dspy.Signature):
    """Responde la duda del usuario usando la definición del glosario provista."""
    glosario_info: str = dspy.InputField(desc="Definición del término del glosario")
    pregunta: str = dspy.InputField()
    respuesta: str = dspy.OutputField()

# En tu código ejecutas la función primero y le pasas el texto limpio al LLM:
definicion = consultar_glosario("deposito")
predictor = dspy.ChainOfThought(AsistenteConContexto)
r = predictor(glosario_info=definicion, pregunta="¿Qué significa que sea de tres meses?")

In [22]:
# --- PASO CLAVE: AUDITAR EL TEXTO EXACTO DE LAS 8 LLAMADAS (CORREGIDO) ---
print("\n=======================================================")
print("   AUDITORÍA DE PASOS: ¿QUÉ HIZO LLAMA 3.2 EN CADA LLAMADA?")
print("=======================================================")

for i, llamada in enumerate(dspy.settings.lm.history):
    print(f"\n🟢 [LLAMADA {i + 1}]")
    
    # 1. Mostrar tokens de este paso
    metadata = llamada.get('response', None)
    if metadata and hasattr(metadata, 'usage') and metadata.usage:
        print(f"   📊 Tokens: Prompt={metadata.usage.prompt_tokens} | Gen={metadata.usage.completion_tokens}")
    
    print("   ------------------------------------------------")
    
    # 2. Qué le envió DSPy al modelo (El Prompt / Mensajes)
    print("   📝 LO QUE RECIBIÓ EL MODELO (Conversación acumulada):")
    
    prompt_completo = llamada.get('prompt', '')
    mensajes = llamada.get('messages', None)
    
    if prompt_completo:  # Formato antiguo / texto plano
        print(f"   ... {prompt_completo[-400:].strip()}")
    elif mensajes and isinstance(mensajes, list):  # Formato moderno / lista de mensajes
        # Tomamos el último mensaje enviado por el usuario para ver qué se acumuló
        ultimo_msg = mensajes[-1].get('content', '')
        print(f"   [Último Mensaje del Historial]:\n   {ultimo_msg[-400:].strip()}")
    else:
        print("   No se pudo leer el prompt de entrada.")
    
    print("   ------------------------------------------------")
    
    # 3. Qué respondió el modelo (Su razonamiento o acción)
    respuesta_modelo = ""
    
    # Intentar extraer desde objeto moderno de LiteLLM
    if metadata and hasattr(metadata, 'choices') and metadata.choices:
        try:
            respuesta_modelo = metadata.choices[0].message.content
        except (TypeError, IndexError):
            try:
                respuesta_modelo = metadata.choices.message.content
            except AttributeError:
                pass
    # Intentar extraer si viene como diccionario plano
    elif isinstance(metadata, dict) and 'choices' in metadata and metadata['choices']:
        respuesta_modelo = metadata['choices'][0].get('message', {}).get('content', '')
    
    # Si todo falla, intentamos sacarlo de las salidas alternativas de la llamada
    if not respuesta_modelo:
        respuesta_modelo = llamada.get('output', '')

    print("   🤖 LO QUE RESPONDIÓ EL MODELO:")
    print(f"   {str(respuesta_modelo).strip()}")
    print("=======================================================")



   AUDITORÍA DE PASOS: ¿QUÉ HIZO LLAMA 3.2 EN CADA LLAMADA?

🟢 [LLAMADA 1]
   📊 Tokens: Prompt=671 | Gen=72
   ------------------------------------------------
   📝 LO QUE RECIBIÓ EL MODELO (Conversación acumulada):
   [Último Mensaje del Historial]:
   de tres meses?

[[ ## trajectory ## ]]


Respond with the corresponding output fields, starting with the field `[[ ## next_thought ## ]]`, then `[[ ## next_tool_name ## ]]` (must be formatted as a valid Python Literal['consultar_glosario', 'finish']), then `[[ ## next_tool_args ## ]]` (must be formatted as a valid Python dict[str, Any]), and then ending with the marker for `[[ ## completed ## ]]`.
   ------------------------------------------------
   🤖 LO QUE RESPONDIÓ EL MODELO:
   [[ next_thought ]]
El depósito de tres meses significa que el inquilino debe mantener un depósito equivalente al primer mes del alquiler.

[[ consultar_glosario | {'termino': 'deposito', 'type': 'string'} ]]

{ "type": "object", "additionalProperties": tru

El motivo real del bucle se reduce a lo siguiente:

1. El Error Técnico Principal: Alucinación de Argumentos ("type": "string")

Mirá con atención las respuestas del modelo en la Llamada 1, Llamada 2 y Llamada 4.
Llama 3.2 intentó llamar a tu función consultar_glosario enviándole esto:
• {"termino": "deposito", "type": "string"}
El modelo inventó un parámetro llamado type que tu función de Python no tiene (tu función solo acepta termino). Como consecuencia, en las llamadas 3 y 4 podés ver que el motor interno de DSPy le devolvió al modelo un error de ejecución explícito:
ValueError: Arg type is not in the tool's args.
Como Llama 3.2 no entendía por qué fallaba la herramienta, siguió intentando inventar parámetros y estructuras en las llamadas siguientes para corregirse, agrandando el prompt infinitamente.

2. Confusión de Formato (JSON vs Corchetes [[ ## ]])

Notá cómo DSPy cambia drásticamente la estrategia en cada llamada para intentar ayudar al modelo:
• En la Llamada 1, DSPy le pide responder usando etiquetas con numerales ([[ ## next_thought ## ]]). El modelo respondió rompiendo los numerales ([[ next_thought ]]).
• En la Llamada 2, al ver que el modelo falló el formato, DSPy cambia las reglas de juego sobre la marcha y le pide un objeto JSON plano ({"next_thought": ...}).
• En la Llamada 3, vuelve a pedirle etiquetas [[ ## ]].
Este "tire y afloje" de formatos volvió loco a un modelo de 3B parámetros como Llama 3.2. Terminó respondiendo cosas híbridas como [[ next_thought_0 ## ]] (agregando ceros aleatorios que DSPy no sabe parsear).

¿Cómo solucionarlo definitivamente?

Para que un modelo local como Llama 3.2 no rompa la lógica de herramientas (Tools), necesitás simplificarle el entorno de dos maneras alternativas:

Opción A: Declarar los tipos con Pydantic para "blindar" la herramienta (Recomendado)

A los modelos chicos les cuesta leer las anotaciones nativas de typing de Python (como Literal[...]). Si usás una clase de Pydantic para estructurar los argumentos de la función, DSPy le va a generar un esquema JSON ultra estricto que Llama 3.2 no va a poder romper.
Modificá la definición de tu herramienta así:

In [23]:
from pydantic import BaseModel, Field

# Creamos un esquema estricto de entrada para la herramienta
# Esto evita que el modelo intente inyectar el parámetro 'type': 'string'
class FiltroGlosario(BaseModel):
    termino: Literal["deposito", "expensas", "rescision", "preaviso"] = Field(
        description="El término exacto que se quiere buscar en el glosario"
    )

def consultar_glosario(arg: FiltroGlosario) -> str:
    """Busca definiciones de términos de contratos de alquiler."""
    # Accedemos al string validado
    termino = arg.termino 
    
    glosario = {
        "deposito": "Suma que garantiza el contrato; suele devolverse al finalizar si no hay deudas ni daños.",
        "expensas": "Gastos del edificio. Ordinarias = mantenimiento habitual; extraordinarias = obras mayores.",
        "rescision": "Finalización anticipada del contrato por una de las partes.",
        "preaviso": "Aviso previo obligatorio antes de terminar o modificar el contrato.",
    }
    return glosario.get(termino, "Término no encontrado.")


In [24]:
# Forzamos a que use JSON puro en cada interacción del agente
agente = dspy.ReAct(AsistenteAlquiler, tools=[consultar_glosario])
# Al correrlo, le pasamos una directiva de formato por si acaso
r = agente(
    pregunta="Según la cláusula, ¿qué significa que el depósito sea de tres meses?",
    config={"max_iters": 2} # Forzamos a que muera en 2 pasos si se equivoca
)


In [25]:
# =======================================================
#               RESULTADOS FINALES DEL AGENTE
# =======================================================

print("\n🏠 RESPUESTA DEL ASISTENTE:")
print("------------------------------------------------")
# r.respuesta contiene el texto final limpio generado para el usuario
print(r.respuesta)
print("------------------------------------------------")

print("\n💰 RESUMEN GENERAL DE CONSUMO:")
# r.get_lm_usage() te da el diccionario total acumulado
uso_total = r.get_lm_usage()
for modelo, conteo in uso_total.items():
    print(f"🔹 Modelo: {modelo}")
    print(f"   - Tokens de entrada (Prompt): {conteo['prompt_tokens']}")
    print(f"   - Tokens generados (Output): {conteo['completion_tokens']}")
    print(f"   - Costo total en tokens: {conteo['total_tokens']}")



🏠 RESPUESTA DEL ASISTENTE:
------------------------------------------------
El depósito de tres meses se refiere al pago que el inquilino debe realizar al final del contrato cuando decide dejar la propiedad.
------------------------------------------------

💰 RESUMEN GENERAL DE CONSUMO:
🔹 Modelo: ollama/llama3.2:latest
   - Tokens de entrada (Prompt): 28994
   - Tokens generados (Output): 1214
   - Costo total en tokens: 30208


¡Es impactante la diferencia! Consumió 30,208 tokens (un salto gigante comparado con los ~8,700 iniciales) para darte una respuesta corta. Además, la respuesta contiene un error de concepto contractual, ya que el depósito se paga al inicio del contrato como garantía, no al final.
Este comportamiento confirma el peor escenario de un bucle de agentes: Llama 3.2 de 3B parámetros se quedó atrapado en un laberinto mental intentando resolver el error del parámetro "type": "string", repitiendo llamadas una y otra vez (llegando casi a los 29,000 tokens de puro historial acumulado) hasta que el bucle se cortó por límite de intentos y te devolvió lo primero que pudo procesar.
Para solucionar esto de raíz, bajar el costo a menos de 1,000 tokens y corregir la precisión de la respuesta, acá tenés el código definitivo unificado utilizando la estrategia de Pydantic para blindar la herramienta y forzando al agente a no dar tantas vueltas:

In [26]:
import dspy
from typing import Literal
from pydantic import BaseModel, Field

# 1. Configuración de red y apagado estricto de caché
dspy.configure_cache(enable_disk_cache=False, enable_memory_cache=False)
lm = dspy.LM(
    model='ollama/llama3.2:latest', 
    api_base='http://ollama:11434'
)
dspy.configure(lm=lm, track_usage=True)

# 2. DEFINICIÓN DEL FILTRO CON PYDANTIC
# Esto obliga a Llama 3.2 a enviar ÚNICAMENTE el parámetro 'termino', eliminando el error del bucle
class EsquemaGlosario(BaseModel):
    termino: Literal["deposito", "expensas", "rescision", "preaviso"] = Field(
        description="El término legal exacto del contrato de alquiler que necesitas consultar."
    )

def consultar_glosario(arg: EsquemaGlosario) -> str:
    """Busca definiciones de términos de contratos de alquiler."""
    glosario = {
        "deposito": "Suma que garantiza el contrato; suele devolverse al finalizar si no hay deudas ni daños.",
        "expensas": "Gastos del edificio. Ordinarias = mantenimiento habitual; extraordinarias = obras mayores.",
        "rescision": "Finalización anticipada del contrato por una de las partes.",
        "preaviso": "Aviso previo obligatorio antes de terminar o modificar el contrato.",
    }
    # Extraemos de forma segura el valor validado por Pydantic
    return glosario.get(arg.termino, "Término no encontrado en el glosario.")

# 3. FIRMA DEL ASISTENTE
class AsistenteAlquiler(dspy.Signature):
    """Responde dudas sobre cláusulas de alquiler usando el glosario obligatorio si el usuario pregunta por un término específico."""
    pregunta: str = dspy.InputField()
    respuesta: str = dspy.OutputField()

# 4. AGENTE REACT LIMITADO
# Ponemos max_iters=2 para que si el modelo se confunde, se detenga de inmediato y no gaste de más
agente = dspy.ReAct(AsistenteAlquiler, tools=[consultar_glosario], max_iters=2)

# 5. EJECUCIÓN
r = agente(
    pregunta="Según la cláusula, ¿qué significa que el depósito sea de tres meses?"
)

In [27]:
# 6. IMPRESIÓN LIMPIA DE RESULTADOS
print("\n🏠 RESPUESTA DEL ASISTENTE:")
print("------------------------------------------------")
print(r.respuesta)
print("------------------------------------------------")

print("\n💰 RESUMEN GENERAL DE CONSUMO:")
uso_total = r.get_lm_usage()
for modelo, conteo in uso_total.items():
    print(f"🔹 Modelo: {modelo}")
    print(f"   - Tokens de entrada (Prompt): {conteo['prompt_tokens']}")
    print(f"   - Tokens generados (Output): {conteo['completion_tokens']}")
    print(f"   - Costo total en tokens: {conteo['total_tokens']}")



🏠 RESPUESTA DEL ASISTENTE:
------------------------------------------------
El depósito de tres meses en una cláusula de alquiler significa el período durante el cual el propietario mantiene posesión del bien y tiene derecho a recibir el pago de la renta y el valor del inmueble en caso de rescisión del contrato.
------------------------------------------------

💰 RESUMEN GENERAL DE CONSUMO:
🔹 Modelo: ollama/llama3.2:latest
   - Tokens de entrada (Prompt): 3715
   - Tokens generados (Output): 429
   - Costo total en tokens: 4144


In [28]:
print("\n=======================================================")
print("   AUDITORÍA DE PROMPTS DE LA CORRIDA OPTIMIZADA")
print("=======================================================")

for i, llamada in enumerate(dspy.settings.lm.history[-2:]): # Miramos las últimas 2 llamadas
    print(f"\n🟢 [LLAMADA {i + 1}]")
    mensajes = llamada.get('messages', None)
    if mensajes and isinstance(mensajes, list):
        # Imprime las instrucciones finales y cómo interactuó el modelo
        print("   📝 CONVERSACIÓN / PROMPT ENVIADO:")
        for msg in mensajes[-2:]: # Mostramos los últimos dos turnos del prompt
            print(f"   [{msg['role'].upper()}]: {msg['content'][-300:].strip()}...\n")
            
    # Qué respondió el modelo en este paso
    metadata = llamada.get('response', None)
    respuesta_modelo = ""
    if metadata and hasattr(metadata, 'choices') and metadata.choices:
        respuesta_modelo = metadata.choices[0].message.content if isinstance(metadata.choices, list) else metadata.choices.message.content
    print(f"   🤖 RESPUESTA INTERNA DEL MODELO:\n   {respuesta_modelo.strip()}")
    print("=======================================================")



   AUDITORÍA DE PROMPTS DE LA CORRIDA OPTIMIZADA

🟢 [LLAMADA 1]
   📝 CONVERSACIÓN / PROMPT ENVIADO:
   [SYSTEM]: nish, whose description is <desc>Marks the task as complete. That is, signals that all information for producing the outputs, i.e. `respuesta`, are now available to be extracted.</desc>. It takes arguments {}.
        When providing `next_tool_args`, the value inside the field must be in JSON format...

   [USER]: the field `[[ ## next_thought ## ]]`, then `[[ ## next_tool_name ## ]]` (must be formatted as a valid Python Literal['consultar_glosario', 'finish']), then `[[ ## next_tool_args ## ]]` (must be formatted as a valid Python dict[str, Any]), and then ending with the marker for `[[ ## completed ## ]]`....

   🤖 RESPUESTA INTERNA DEL MODELO:
   [[ ## next_thought ## ]]
¿El depósito de tres meses en una cláusula de alquiler significa el período durante el cual el propietario mantiene posesión del bien?

[[ ## next_tool_name ## ]]
consultar_glosario

[[ ## next_tool_args

In [29]:
import dspy
from typing import Literal

# 1. Configuración limpia de Ollama (Sin caché)
dspy.configure_cache(enable_disk_cache=False, enable_memory_cache=False)
lm = dspy.LM('ollama/llama3.2:latest', api_base='http://ollama:11434')
dspy.configure(lm=lm, track_usage=True)

# 2. Tu glosario nativo de Python (Sin Pydantic, no hace falta)
def consultar_glosario(termino: str) -> str:
    glosario = {
        "deposito": "Suma que garantiza el contrato; suele devolverse al finalizar si no hay deudas ni daños.",
        "expensas": "Gastos del edificio. Ordinarias = mantenimiento habitual; extraordinarias = obras mayores.",
        "rescision": "Finalización anticipada del contrato por una de las partes.",
        "preaviso": "Aviso previo obligatorio antes de terminar o modificar el contrato.",
    }
    return glosario.get(termino.lower(), "Término no encontrado en el glosario.")

# 3. Una firma lineal donde le INYECTAMOS el contexto masticado al modelo
class AsistenteAlquilerEstatico(dspy.Signature):
    """Responde la duda de un contrato de alquiler basándote estrictamente 
    en la información legal provista en el campo 'contexto_glosario'.
    """
    contexto_glosario: str = dspy.InputField(desc="Definición oficial extraída del glosario legal")
    pregunta: str = dspy.InputField(desc="Pregunta del inquilino")
    respuesta: str = dspy.OutputField(desc="Explicación clara y legalmente correcta")

# 4. Usamos ChainOfThought (obliga al modelo a pensar antes de responder, pero sin usar herramientas)
asistente = dspy.ChainOfThought(AsistenteAlquilerEstatico)

# --- FLUJO DE EJECUCIÓN ---
pregunta_usuario = "Según la cláusula, ¿qué significa que el depósito sea de tres meses?"

# Paso A: Python resuelve la lógica pesada (Buscamos 'deposito' de forma nativa)
informacion_legal = consultar_glosario("deposito")

# Paso B: Le pasamos el texto limpio a Llama 3.2 para que solo redacte
r = asistente(
    contexto_glosario=informacion_legal,
    pregunta=pregunta_usuario
)

# 5. Impresión de resultados optimizados
print("\n🏠 RESPUESTA DEL ASISTENTE:")
print("------------------------------------------------")
print(r.respuesta)
print("------------------------------------------------")

print("\n💰 CONSUMO REAL:")
print(r.get_lm_usage())



🏠 RESPUESTA DEL ASISTENTE:
------------------------------------------------
El depósito es una cantidad que se paga junto con el primer mes del contrato y se le devuelve en la misma cantidad una vez que el inquilino haya cumplido su obligations contractuales sin causar daño alguno a los bienes arrendados o de otra manera no haber incurrido en ninguna infracción al contrato.
------------------------------------------------

💰 CONSUMO REAL:
{'ollama/llama3.2:latest': {'completion_tokens': 137, 'prompt_tokens': 320, 'total_tokens': 457, 'completion_tokens_details': None, 'prompt_tokens_details': None}}


In [33]:
import dspy
from dspy.evaluate import Evaluate

# =======================================================
# 1. CONFIGURACIÓN DEL ENTORNO (Ollama Local)
# =======================================================
dspy.configure_cache(enable_disk_cache=False, enable_memory_cache=False)
lm = dspy.LM('ollama/llama3.2:latest', api_base='http://ollama:11434')
dspy.configure(lm=lm, track_usage=True)

# =======================================================
# 2. LOGICA DEL ASISTENTE LINEAL (Tu código optimizado)
# =======================================================
def consultar_glosario(termino: str) -> str:
    glosario = {
        "deposito": "Suma que garantiza el contrato; suele devolverse al finalizar si no hay deudas ni daños.",
        "expensas": "Gastos del edificio. Ordinarias = mantenimiento habitual; extraordinarias = obras mayores.",
        "rescision": "Finalización anticipada del contrato por una de las partes.",
        "preaviso": "Aviso previo obligatorio antes de terminar o modificar el contrato.",
    }
    return glosario.get(termino.lower(), "Término no encontrado.")

class AsistenteAlquilerEstatico(dspy.Signature):
    """Responde la duda de un contrato de alquiler basándote estrictamente en el contexto_glosario."""
    contexto_glosario: str = dspy.InputField(desc="Definición oficial del glosario legal")
    pregunta: str = dspy.InputField(desc="Pregunta del inquilino")
    respuesta: str = dspy.OutputField(desc="Explicación clara y legalmente correcta")

asistente = dspy.ChainOfThought(AsistenteAlquilerEstatico)

# Envoltura pequeña para que DSPy pueda iterar el pipeline completo durante la evaluación
class PipelineAsistente(dspy.Module):
    def __init__(self):
        super().__init__()
        self.asistente = dspy.ChainOfThought(AsistenteAlquilerEstatico)
        
    def forward(self, termino_glosario: str, pregunta: str):
        # Python resuelve el contexto primero
        informacion_legal = consultar_glosario(termino_glosario)
        # El LLM redacta la respuesta
        resultado = self.asistente(contexto_glosario=informacion_legal, pregunta=pregunta)
        return dspy.Prediction(respuesta=resultado.respuesta)

pipeline = PipelineAsistente()

# =======================================================
# 3. DATASET DE CONTROL DE CALIDAD (Ground Truth)
# =======================================================
# Definimos preguntas reales, qué término debe buscar y qué palabras clave DEBE contener la respuesta
dataset_control = [
    dspy.Example(
        termino_glosario="deposito",
        pregunta="Según la cláusula, ¿qué significa que el depósito sea de tres meses?",
        keywords_obligatorias=["devol", "garant", "daño"] # Palabras clave esenciales para validar calidad
    ).with_inputs("termino_glosario", "pregunta"),
    
    dspy.Example(
        termino_glosario="expensas",
        pregunta="Me vino una boleta de expensas extraordinarias, ¿las tengo que pagar?",
        keywords_obligatorias=["obra", "mayor", "extraordinaria"]
    ).with_inputs("termino_glosario", "pregunta")
]

# =======================================================
# 4. FUNCIÓN MÉTRICA: CALCULA EL ACCURACY
# =======================================================
def evaluar_respuesta_y_palabras_clave(ejemplo, prediccion, trace=None):
    respuesta_llm = prediccion.respuesta.lower()
    
    # Verificamos si la respuesta del LLM incluyó los conceptos legales obligatorios
    aciertos = [kw in respuesta_llm for kw in ejemplo.keywords_obligatorias]
    
    # Si contiene todas las palabras clave conceptuales, la respuesta es de calidad (1.0), si no (0.0)
    return 1.0 if all(aciertos) else 0.0

# =======================================================
# 5. EJECUCIÓN DEL CONTROL DE CALIDAD Y RATIO (CORREGIDO)
# =======================================================
evaluador = Evaluate(
    devset=dataset_control, 
    metric=evaluar_respuesta_y_palabras_clave, 
    num_threads=1, 
    display_progress=False
)

print("Corriendo el control de calidad en lote...")
# 1. Ejecutar el test sobre el dataset completo (devuelve un EvaluationResult)
resultado_evaluacion = evaluador(pipeline)

# score ya viene en porcentaje (50.0 == 50%). La métrica cruda (0.0 o 1.0) está en results.
accuracy_porcentaje = resultado_evaluacion.score
accuracy_fraccion = accuracy_porcentaje / 100.0

# 2. CÁLCULO SEGURO DE TOKENS TOTALES DESDE EL HISTORIAL GLOBAL
tokens_totales = 0
for llamada in lm.history:
    metadata = llamada.get('response', None)
    if metadata and hasattr(metadata, 'usage') and metadata.usage:
        tokens_totales += metadata.usage.total_tokens

# 3. Calcular el Ratio Calidad-Precio-Token (Puntos de precisión por cada 1,000 tokens)
ratio_eficiencia = (accuracy_fraccion / tokens_totales) * 1000 if tokens_totales > 0 else 0

# =======================================================
# 6. PANEL DE CONTROL FINAL
# =======================================================
print("\n=======================================================")
print("        PANEL DE CONTROL DE CALIDAD / PRECIO / TOKEN     ")
print("=======================================================")
print(f"📊 Precisión del Pipeline (Accuracy): {accuracy_porcentaje:.2f}%")
print(f"🪙 Gasto Total de la Evaluación: {tokens_totales} tokens")
print(f"🎯 Índice de Eficiencia: {ratio_eficiencia:.4f} pts de calidad / 1K tokens")
print("=======================================================")



Corriendo el control de calidad en lote...


2026/10/27 01:15:48 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 2 (0.0%)



        PANEL DE CONTROL DE CALIDAD / PRECIO / TOKEN     
📊 Precisión del Pipeline (Accuracy): 0.00%
🪙 Gasto Total de la Evaluación: 1072 tokens
🎯 Índice de Eficiencia: 0.0000 pts de calidad / 1K tokens
